# Lab 04: What Recurrence Can and Cannot Remember

The task comes from `lab04_starter.py`. The label is only at step 0 and the model has to report it at the last step, so chance is 0.25. Every run uses the starter's defaults (hidden 32, 600 updates, batch 64, Adam lr 3e-3, clipping at 1.0) unless I say otherwise.

In [1]:
import os
import time

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from lab04_starter import RecallNet, make_recall_batch, train_recall

SEEDS = [0, 1, 2]
os.makedirs("figures", exist_ok=True)

## Run the starter once

In [2]:
accuracy, _ = train_recall("rnn", seq_len=10, seed=0)
print(f"plain RNN, 10 steps: {accuracy:.3f}")

plain RNN, 10 steps: 1.000


Got 1.000, so the environment works.

## Check the gate bias lands on the right slice

I print the effective bias (`bias_ih + bias_hh`) for every gate block, to make sure `gate_bias=3` only changes the forget gate in the LSTM and the update gate in the GRU.

In [3]:
def print_gate_blocks(kind, gate_bias):
    rnn = RecallNet(kind, gate_bias=gate_bias).rnn
    H = rnn.hidden_size
    total = (rnn.bias_ih_l0 + rnn.bias_hh_l0).detach()
    names = {"lstm": ["input", "forget", "cell", "output"], "gru": ["reset", "update", "new"]}[kind]
    for i, name in enumerate(names):
        block = total[i * H:(i + 1) * H]
        print(f"{kind} {name:>6} [{i * H}:{(i + 1) * H}]  min {block.min():.3f}  max {block.max():.3f}")


torch.manual_seed(0)
print_gate_blocks("lstm", 3.0)
print()
print_gate_blocks("gru", 3.0)

lstm  input [0:32]  min -0.235  max 0.276
lstm forget [32:64]  min 3.000  max 3.000
lstm   cell [64:96]  min -0.262  max 0.313
lstm output [96:128]  min -0.304  max 0.325

gru  reset [0:32]  min -0.218  max 0.261
gru update [32:64]  min 3.000  max 3.000
gru    new [64:96]  min -0.280  max 0.239


Only `[32:64]` is exactly 3.0 in both cells, and that's the forget gate in the LSTM and the update gate in the GRU. The other blocks still have PyTorch's small random values, around ±0.3.

## Benchmark five configs on lengths 10, 30 and 60

In [4]:
CONFIGS = {
    "RNN": ("rnn", None),
    "LSTM default": ("lstm", None),
    "LSTM forget bias 3": ("lstm", 3.0),
    "GRU default": ("gru", None),
    "GRU update bias 3": ("gru", 3.0),
}


def run_seeds(kind, seq_len, gate_bias=None, clip=1.0):
    return [train_recall(kind, seq_len, seed=s, gate_bias=gate_bias, clip=clip)[0] for s in SEEDS]


def print_row(name, accs):
    seeds = "  ".join(f"{a:.3f}" for a in accs)
    print(f"{name:<20} {seeds}   mean {sum(accs) / len(accs):.3f}")


benchmark = {}
for seq_len in [10, 30, 60]:
    print(f"{'length ' + str(seq_len):<20} seed0  seed1  seed2")
    for name, (kind, bias) in CONFIGS.items():
        benchmark[(name, seq_len)] = run_seeds(kind, seq_len, bias)
        print_row(name, benchmark[(name, seq_len)])
    print()

length 10            seed0  seed1  seed2


RNN                  1.000  1.000  0.495   mean 0.832


LSTM default         1.000  1.000  1.000   mean 1.000


LSTM forget bias 3   1.000  1.000  1.000   mean 1.000


GRU default          1.000  1.000  1.000   mean 1.000


...

RNN                  0.250  0.240  0.233   mean 0.241


LSTM default         0.249  0.251  0.254   mean 0.251


LSTM forget bias 3   1.000  1.000  1.000   mean 1.000


GRU default          0.248  0.249  0.240   mean 0.246


GRU update bias 3    1.000  1.000  1.000   mean 1.000



At length 10 almost everything solves it. The one miss is the plain RNN on seed 2, stuck at 0.495, so even the short case isn't guaranteed for the plain RNN.

At 30 and 60 the plain RNN and both default gated cells sit at chance (0.233 to 0.264) on every seed. Starting the gate open (LSTM forget bias 3, GRU update bias 3) gets 1.000 on every seed at both lengths. The default LSTM and GRU have the same gates and still fail, so having the gates wasn't enough on its own.